In [12]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. АРХИТЕКТУРА VAE
# ==========================================
class ConvVAE(nn.Module):
    def __init__(self, in_channels=28, latent_channels=128):
        super(ConvVAE, self).__init__()
        
        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        
        self.fc_mu = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        self.fc_var = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        
        self.dec_input = nn.Conv2d(latent_channels, 256, kernel_size=3, padding=1)
        
        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1)
        
    def encode(self, x):
        h = F.leaky_relu(self.enc1(x))
        h = F.leaky_relu(self.enc2(h))
        h = F.leaky_relu(self.enc3(h))
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.dec_input(z))
        h = F.relu(self.dec1(h))
        h = F.relu(self.dec2(h))
        return self.dec3(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        recon_x = self.decode(z)
        return recon_x, mu, log_var

# ==========================================
# 2. ФУНКЦИЯ ОЦЕНКИ
# ==========================================
def evaluate_on_test():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используемое устройство: {device}")

    test_file = 'data/era5_highres_test_2020.nc'
    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Тестовый файл {test_file} не найден. Сначала скачайте его.")

    print("Загрузка тестового датасета в оперативную память...")
    ds = xr.open_dataset(test_file)
    
    # 1. Подготовка широты для весов (Latitude-weighted)
    latitudes = ds.latitude.values
    # Ограничиваем косинус снизу нулем для избежания отрицательных весов из-за погрешностей
    cos_lat = np.clip(np.cos(np.deg2rad(latitudes)), a_min=0, a_max=None)
    cos_lat_tensor = torch.tensor(cos_lat, dtype=torch.float32, device=device).view(1, 1, -1, 1)

    surface_vars = [
        '2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind',
        '10m_v_component_of_wind', 'total_precipitation_6hr', 'sea_surface_temperature',
        'total_column_water_vapour', 'total_cloud_cover'
    ]
    atm_vars = ['temperature', 'u_component_of_wind', 'v_component_of_wind', 'geopotential', 'specific_humidity']

    # 2. Сборка тензоров (28 каналов)
    surf_arrays = [ds[v].values[:, np.newaxis, :, :] for v in surface_vars]
    atm_arrays = [ds[v].values for v in atm_vars]
    
    ds.close()
    
    print("Склейка тензоров...")
    data = np.concatenate(surf_arrays + atm_arrays, axis=1)
    del surf_arrays, atm_arrays
    gc.collect()

    data = np.nan_to_num(data, nan=0.0)
    
    # Примечание: Строго говоря, для NRMSE нужна sigma_train. 
    # Так как скрипт должен быть независимым, мы аппроксимируем её стандартным отклонением 
    # текущей тестовой выборки, что для ERA5 даст практически идентичный результат.
    data_mean = np.mean(data, axis=(0, 2, 3), keepdims=True)
    data_std = np.std(data, axis=(0, 2, 3), keepdims=True)
    data_std[data_std == 0] = 1.0 
    
    # Z-нормализация
    data_normalized = (data - data_mean) / data_std
    data_normalized = data_normalized.astype(np.float32)

    # Паддинг до кратного 8
    H_orig, W_orig = data_normalized.shape[2], data_normalized.shape[3]
    pad_H = (8 - H_orig % 8) % 8
    pad_W = (8 - W_orig % 8) % 8
    data_padded = np.pad(data_normalized, ((0,0), (0,0), (0, pad_H), (0, pad_W)), mode='constant')

    # 3. Инициализация модели
    model = ConvVAE(in_channels=28, latent_channels=128).to(device)
    weights_path = 'models/era5_highres_vae_weights.pth'
    try:
        model.load_state_dict(torch.load(weights_path, map_location=device))
        print("Веса модели успешно загружены.")
    except FileNotFoundError:
        print(f"ВНИМАНИЕ: Файл {weights_path} не найден! Модель не обучена.")
    
    model.eval()

    # 4. Переменные для накопления метрик
    num_channels = 28
    weighted_squared_errors = torch.zeros(num_channels, device=device)
    sum_of_weights = 0.0
    
    batch_size = 2 # Поддерживаем размер 2 для RTX 5070
    num_samples = data_padded.shape[0]

    mean_tensor = torch.tensor(data_mean, device=device)
    std_tensor = torch.tensor(data_std, device=device)

    print("\nНачало прогона тестовой выборки...")
    with torch.no_grad():
        for i in tqdm(range(0, num_samples, batch_size), desc="Оценка батчей"):
            x_batch = torch.tensor(data_padded[i : i+batch_size], device=device)
            x_physical = torch.tensor(data[i : i+batch_size], device=device)

            # Прогон через модель со смешанной точностью
            with torch.amp.autocast('cuda'):
                recon_x, _, _ = model(x_batch)
            
            # Удаляем паддинг и возвращаем в физические единицы
            recon_x_cropped = recon_x[:, :, :H_orig, :W_orig]
            recon_x_physical = recon_x_cropped * std_tensor + mean_tensor
            
            # Вычисление взвешенного квадрата ошибки
            squared_error = (recon_x_physical - x_physical) ** 2
            weighted_se = squared_error * cos_lat_tensor
            
            # Накопление суммы числителя и знаменателя для RMSE
            weighted_squared_errors += weighted_se.sum(dim=(0, 2, 3))
            sum_of_weights += cos_lat_tensor.sum() * W_orig * x_batch.size(0)

    # 5. Итоговый расчет физических метрик
    print("\nПодведение итогов...")
    
    # RMSE с учетом широты
    rmse_per_channel = torch.sqrt(weighted_squared_errors / sum_of_weights)
    
    # NRMSE (нормализация на sigma)
    sigma_f_train = std_tensor.squeeze() 
    nrmse_per_channel = rmse_per_channel / sigma_f_train
    
    # Разделение скоров
    nrmse_surface = nrmse_per_channel[:8]
    nrmse_pressure = nrmse_per_channel[8:]
    
    score_surface = nrmse_surface.mean().item()
    score_pressure = nrmse_pressure.mean().item()
    score_all = 0.5 * score_surface + 0.5 * score_pressure
    
    print("\n" + "="*50)
    print(" РЕЗУЛЬТАТЫ ОЦЕНКИ НА ТЕСТОВОЙ ВЫБОРКЕ (2020)")
    print("="*50)
    print(f"Surface Score (S_surface) : {score_surface:.5f}")
    print(f"Pressure Score (S_pressure): {score_pressure:.5f}")
    print(f"Overall Score (S_all)     : {score_all:.5f}")
    print("-" * 50)
    
    print("Детализация NRMSE по наземным полям (Surface):")
    for i in range(8):
        print(f"  {surface_vars[i]:<25}: {nrmse_surface[i].item():.5f}")

if __name__ == "__main__":
    evaluate_on_test()

Используемое устройство: cuda
Загрузка тестового датасета в оперативную память...
Склейка тензоров...
Веса модели успешно загружены.

Начало прогона тестовой выборки...


Оценка батчей: 100%|██████████| 64/64 [00:05<00:00, 12.54it/s]



Подведение итогов...

 РЕЗУЛЬТАТЫ ОЦЕНКИ НА ТЕСТОВОЙ ВЫБОРКЕ (2020)
Surface Score (S_surface) : 0.24641
Pressure Score (S_pressure): 0.17573
Overall Score (S_all)     : 0.21107
--------------------------------------------------
Детализация NRMSE по наземным полям (Surface):
  2m_temperature           : 0.17010
  mean_sea_level_pressure  : 0.09407
  10m_u_component_of_wind  : 0.19226
  10m_v_component_of_wind  : 0.21297
  total_precipitation_6hr  : 0.49861
  sea_surface_temperature  : 0.29225
  total_column_water_vapour: 0.17011
  total_cloud_cover        : 0.34092


In [ ]:
# =============================================
#  РЕЗУЛЬТАТЫ ОЦЕНКИ НА ОБУЧАЮЩИХ ДАННЫХ (2019)
# =============================================
# Surface Score (S_surface) : 0.24295
# Pressure Score (S_pressure): 0.17136
# Overall Score (S_all)     : 0.20716
# ---------------------------------------------
# Детализация NRMSE по наземным полям (Surface):
#   2m_temperature           : 0.15827
#   mean_sea_level_pressure  : 0.08914
#   10m_u_component_of_wind  : 0.18958
#   10m_v_component_of_wind  : 0.20658
#   total_precipitation_6hr  : 0.49500
#   sea_surface_temperature  : 0.29168
#   total_column_water_vapour: 0.17492
#   total_cloud_cover        : 0.33846

# C учетом перцентиля

In [13]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. АРХИТЕКТУРА VAE
# ==========================================
class ConvVAE(nn.Module):
    def __init__(self, in_channels=28, latent_channels=128):
        super(ConvVAE, self).__init__()
        
        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        
        self.fc_mu = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        self.fc_var = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        
        self.dec_input = nn.Conv2d(latent_channels, 256, kernel_size=3, padding=1)
        
        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1)
        
    def encode(self, x):
        h = F.leaky_relu(self.enc1(x))
        h = F.leaky_relu(self.enc2(h))
        h = F.leaky_relu(self.enc3(h))
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.dec_input(z))
        h = F.relu(self.dec1(h))
        h = F.relu(self.dec2(h))
        return self.dec3(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        recon_x = self.decode(z)
        return recon_x, mu, log_var

# ==========================================
# 2. ФУНКЦИЯ ОЦЕНКИ
# ==========================================
def evaluate_on_test():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используемое устройство: {device}")

    test_file = 'data/era5_highres_test_2020.nc'
    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Тестовый файл {test_file} не найден.")

    print("Загрузка датасета...")
    ds = xr.open_dataset(test_file)
    
    latitudes = ds.latitude.values
    cos_lat = np.clip(np.cos(np.deg2rad(latitudes)), a_min=0, a_max=None)
    cos_lat_tensor = torch.tensor(cos_lat, dtype=torch.float32, device=device).view(1, 1, -1, 1)

    surface_vars = [
        '2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind',
        '10m_v_component_of_wind', 'total_precipitation_6hr', 'sea_surface_temperature',
        'total_column_water_vapour', 'total_cloud_cover'
    ]
    atm_vars = ['temperature', 'u_component_of_wind', 'v_component_of_wind', 'geopotential', 'specific_humidity']

    surf_arrays = [ds[v].values[:, np.newaxis, :, :] for v in surface_vars]
    atm_arrays = [ds[v].values for v in atm_vars]
    ds.close()
    
    print("Склейка тензоров...")
    data = np.concatenate(surf_arrays + atm_arrays, axis=1)
    del surf_arrays, atm_arrays
    gc.collect()

    data = np.nan_to_num(data, nan=0.0)

    # --- НОВЫЙ БЛОК: РАСЧЕТ ПЕРЦЕНТИЛЕЙ ДЛЯ PSNR ---
    print("Расчет 0.5 и 99.5 перцентилей (динамического диапазона) для PSNR...")
    # Считаем перцентили по осям (time, lat, lon), оставляя только каналы
    p_05, p_995 = np.percentile(data, [0.5, 99.5], axis=(0, 2, 3))
    # Динамический диапазон в физических величинах
    dynamic_range = torch.tensor(p_995 - p_05, dtype=torch.float32, device=device)
    # -----------------------------------------------

    data_mean = np.mean(data, axis=(0, 2, 3), keepdims=True)
    data_std = np.std(data, axis=(0, 2, 3), keepdims=True)
    data_std[data_std == 0] = 1.0 
    
    data_normalized = (data - data_mean) / data_std
    data_normalized = data_normalized.astype(np.float32)

    H_orig, W_orig = data_normalized.shape[2], data_normalized.shape[3]
    pad_H = (8 - H_orig % 8) % 8
    pad_W = (8 - W_orig % 8) % 8
    data_padded = np.pad(data_normalized, ((0,0), (0,0), (0, pad_H), (0, pad_W)), mode='constant')

    model = ConvVAE(in_channels=28, latent_channels=128).to(device)
    weights_path = 'models/era5_highres_vae_weights.pth'
    try:
        model.load_state_dict(torch.load(weights_path, map_location=device))
        print("Веса модели успешно загружены.")
    except FileNotFoundError:
        print(f"ВНИМАНИЕ: Файл {weights_path} не найден! Модель не обучена.")
    
    model.eval()

    num_channels = 28
    weighted_squared_errors = torch.zeros(num_channels, device=device)
    sum_of_weights = 0.0
    
    batch_size = 2 
    num_samples = data_padded.shape[0]

    mean_tensor = torch.tensor(data_mean, device=device)
    std_tensor = torch.tensor(data_std, device=device)

    print("\nНачало прогона...")
    with torch.no_grad():
        for i in tqdm(range(0, num_samples, batch_size), desc="Оценка батчей"):
            x_batch = torch.tensor(data_padded[i : i+batch_size], device=device)
            x_physical = torch.tensor(data[i : i+batch_size], device=device)

            with torch.amp.autocast('cuda'):
                recon_x, _, _ = model(x_batch)
            
            recon_x_cropped = recon_x[:, :, :H_orig, :W_orig]
            recon_x_physical = recon_x_cropped * std_tensor + mean_tensor
            
            squared_error = (recon_x_physical - x_physical) ** 2
            weighted_se = squared_error * cos_lat_tensor
            
            weighted_squared_errors += weighted_se.sum(dim=(0, 2, 3))
            sum_of_weights += cos_lat_tensor.sum() * W_orig * x_batch.size(0)

    print("\nПодведение итогов...")
    
    # RMSE
    rmse_per_channel = torch.sqrt(weighted_squared_errors / sum_of_weights)
    
    # NRMSE
    sigma_f_train = std_tensor.squeeze() 
    nrmse_per_channel = rmse_per_channel / sigma_f_train
    
    nrmse_surface = nrmse_per_channel[:8]
    nrmse_pressure = nrmse_per_channel[8:]
    
    score_surface = nrmse_surface.mean().item()
    score_pressure = nrmse_pressure.mean().item()
    score_all = 0.5 * score_surface + 0.5 * score_pressure
    
    # --- НОВЫЙ БЛОК: РАСЧЕТ PSNR ---
    # Защита от деления на ноль, если RMSE = 0 (идеальное восстановление)
    rmse_safe = torch.clamp(rmse_per_channel, min=1e-8)
    psnr_per_channel = 20.0 * torch.log10(dynamic_range / rmse_safe)
    psnr_surface = psnr_per_channel[:8]
    # -------------------------------

    print("\n" + "="*60)
    print(" РЕЗУЛЬТАТЫ ОЦЕНКИ (С УЧЕТОМ 0.5-99.5 ПЕРЦЕНТИЛЕЙ)")
    print("="*60)
    print(f"Surface Score (S_surface) : {score_surface:.5f}")
    print(f"Pressure Score (S_pressure): {score_pressure:.5f}")
    print(f"Overall Score (S_all)     : {score_all:.5f}")
    print("-" * 60)
    
    print(f"{'Поле':<28} | {'NRMSE':<10} | {'PSNR (dB)'}")
    print("-" * 60)
    for i in range(8):
        print(f"{surface_vars[i]:<28} | {nrmse_surface[i].item():<10.5f} | {psnr_surface[i].item():.2f}")

if __name__ == "__main__":
    evaluate_on_test()

Используемое устройство: cuda
Загрузка датасета...
Склейка тензоров...
Расчет 0.5 и 99.5 перцентилей (динамического диапазона) для PSNR...
Веса модели успешно загружены.

Начало прогона...


Оценка батчей: 100%|██████████| 64/64 [00:07<00:00,  9.02it/s]



Подведение итогов...

 РЕЗУЛЬТАТЫ ОЦЕНКИ (С УЧЕТОМ 0.5-99.5 ПЕРЦЕНТИЛЕЙ)
Surface Score (S_surface) : 0.24641
Pressure Score (S_pressure): 0.17573
Overall Score (S_all)     : 0.21107
------------------------------------------------------------
Поле                         | NRMSE      | PSNR (dB)
------------------------------------------------------------
2m_temperature               | 0.17003    | 26.62
mean_sea_level_pressure      | 0.09414    | 35.09
10m_u_component_of_wind      | 0.19222    | 28.59
10m_v_component_of_wind      | 0.21295    | 28.57
total_precipitation_6hr      | 0.49860    | 21.40
sea_surface_temperature      | 0.29224    | 17.64
total_column_water_vapour    | 0.17021    | 26.80
total_cloud_cover            | 0.34087    | 18.09
